# **ILP Meal Planner v2**

This notebook tests the `ILPMealPlanner` v2, which uses Integer Linear Programming (via [SciPy `milp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html) and the HiGHS backend) to find a provably near-optimal 7-day meal plan.

The ILP maximises the same composite objective as the GA:
$$\text{fitness} = \text{pantry\_score} - \text{dietary\_penalty} - \text{waste\_penalty} - \text{budget\_penalty}$$

It serves as an **oracle / upper bound** when evaluating other planners — any other planner's fitness score can be compared against the ILP to compute an *optimality gap*.

In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random, seed

from engines import (
    filter_and_add_recipes,
    get_pantry_ingredient,
    load_all_ingredients,
    make_preferences,
    parse_recipes,
)
from engines.ILPMealPlanner import ILPMealPlanner
from models import MealPlanningEnvironment, Pantry

In [2]:
seed(1)

In [3]:
preferences = make_preferences()

In [4]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [5]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [10]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-05-06
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-09
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-11
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
with open("../recipe_extraction/supplemented_structured_recipes.json", "r") as file:
    all_recipes = json.load(file)

In [12]:
all_recipes = parse_recipes(all_recipes)

In [13]:
NUM_FILTERED_RECIPES = 100
NUM_EXTRA_RECIPES = 50

In [14]:
final_recipes = filter_and_add_recipes(
    all_recipes, [ingredient.name for ingredient in pantry_ingredients], NUM_FILTERED_RECIPES, NUM_EXTRA_RECIPES
)

In [15]:
print(len(final_recipes))

150


In [16]:
print(f"Number of recipes before filtering: {len(all_recipes)}")
print(f"Number of recipes after filtering: {len(final_recipes)}")

Number of recipes before filtering: 19716
Number of recipes after filtering: 150


In [17]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=final_recipes,
    pantry=pantry,
    preferences=preferences,
)

In [18]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [19]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [20]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

In [21]:
planner = ILPMealPlanner(meal_planning_environment)

In [22]:
best_meal_plan, fitness_score = planner.generate_meal_plan(msg=True)

In [23]:
print(f"Solve status : {planner.get_solve_status()}")
print(f"Fitness score: {fitness_score:.4f}")

Solve status : Optimal
Fitness score: -0.0332


In [24]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,226.796185,573.203815,1d
1,broccoli,1500,1500.000000,0.000000,4d
2,rice,1000,875.466667,124.533333,6d


In [25]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Meal 1,Meal 2,Meal 3
Day 1,Roast Turkey with Pesto-Rice Stuffing,Chicken Tortilla Soup,Chicken Tortilla Soup
Day 2,Moroccan Chicken with Kumquats and Prunes,Cream of Broccoli Soup,Roasted Broccoli Florets with Toasted Breadcru...
Day 3,Moroccan Chicken with Kumquats and Prunes,Roasted Veg With Nutritional Yeast,Roasted Broccoli Florets with Toasted Breadcru...
Day 4,Moroccan Chicken with Kumquats and Prunes,Cream of Broccoli Soup,Roasted Broccoli Florets with Toasted Breadcru...
Day 5,"Fried Rice with Ham, Egg, and Scallions","Fried Rice with Ham, Egg, and Scallions",Moroccan Chicken with Kumquats and Prunes
Day 6,"Fried Rice with Ham, Egg, and Scallions","Fried Rice with Ham, Egg, and Scallions",Moroccan Chicken with Kumquats and Prunes
Day 7,Roast Turkey with Pesto-Rice Stuffing,Beef and Broccoli Stir Fry,"Coconut, Beet, and Ginger Soup"


In [26]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: \u20ac{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 63
Total cost: €63.98


,Ingredient,Quantity to Buy (g),Cost (€)
0,Bacon slices,7.4,0.03
1,Chicken Bone Broth,33.8,0.09
2,Crushed tortilla chips,50.0,0.17
3,Kosher salt,100.0,0.03
4,Olive oil,7.4,0.01
...,...,...,...
59,white rice,25.0,0.14
60,white wine,5.6,0.00
61,white wine vinegar,0.7,0.00
62,whole beets,43.2,0.11


In [27]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,2270.1 kcal,53.9 g,270.1 kcal,3.9 g
Day 2,1979.5 kcal,51.1 g,-20.5 kcal,1.1 g
Day 3,1971.6 kcal,53.2 g,-28.4 kcal,3.2 g
Day 4,1979.5 kcal,51.1 g,-20.5 kcal,1.1 g
Day 5,2097.8 kcal,47.5 g,97.8 kcal,-2.5 g
Day 6,2097.8 kcal,47.5 g,97.8 kcal,-2.5 g
Day 7,1974.8 kcal,52.3 g,-25.2 kcal,2.3 g
